# Cascading example

A minimal, self-contained walk-through of *cascading* a loading matrix: re-attribute every illiquid tenor's risk down the liquidity layers onto the first (most liquid) layer, so the position can be hedged with liquid instruments.

The trick: with rows **and** columns both in layer-then-maturity order, a loading matrix is block-lower-triangular. Raising it to the power `(#layers - 1)` folds every intermediate layer through to the first, leaving only the first `k` columns nonzero (`k` = number of first-layer anchors).

In [ ]:
import numpy as np
import pandas as pd

## 1. A toy loading matrix

A tiny 3-layer curve, with tenors listed in layer-then-maturity order. Each **row** is a tenor written as a combination of more-liquid tenors (which sit earlier in `order`): anchor rows are the identity, and each deeper-layer row loads only on earlier layers. So this matrix is block-lower-triangular.

In [ ]:
order  = ["2Y", "10Y", "5Y", "3Y", "7Y"]   # layer-then-maturity
layers = [0, 0, 1, 2, 2]                    # liquidity layer of each tenor
k      = layers.count(0)                    # first-layer anchors
p      = max(layers)                        # = #layers - 1 cascade steps

# Row = tenor expressed as a combo of more-liquid tenors (earlier in `order`).
M = np.array([
    # 2Y   10Y   5Y    3Y   7Y
    [1.0,  0.0,  0.0,  0.0, 0.0],   # 2Y  anchor
    [0.0,  1.0,  0.0,  0.0, 0.0],   # 10Y anchor
    [0.5,  0.5,  0.0,  0.0, 0.0],   # 5Y  = 0.5*2Y + 0.5*10Y
    [0.4,  0.0,  0.6,  0.0, 0.0],   # 3Y  = 0.4*2Y + 0.6*5Y
    [0.0,  0.3,  0.7,  0.0, 0.0],   # 7Y  = 0.3*10Y + 0.7*5Y
])
pd.DataFrame(M, index=order, columns=order)

## 2. Cascade by raising to a power

One application of `M` rewrites each tenor one layer more liquid (e.g. `3Y` still references `5Y`). Applying it `p` times pushes every reference onto layer 0, so the columns for non-anchor tenors collapse to exactly zero.

In [ ]:
cascaded = np.linalg.matrix_power(M, p)

print(f"k = {k} anchors, p = {p} cascade steps")
print("max |loading| in non-anchor columns:", np.abs(cascaded[:, k:]).max())  # -> 0
pd.DataFrame(cascaded, index=order, columns=order).round(3)

## 3. First-layer attribution

Drop the anchor rows and the now-zero non-anchor columns. Each remaining row is an illiquid tenor replicated purely by the liquid anchors — i.e. its hedge. (Here every row sums to 1, since each original loading row did.)

This is exactly what the dashboard's **Cascading** panel shows. `sequential_pca` is the one exception: its first-layer factor columns already hold this attribution, so it slices them directly instead of taking a power.

In [ ]:
attribution = pd.DataFrame(cascaded[k:, :k], index=order[k:], columns=order[:k])
attribution.round(3)